# Peer Group Analysis

Goal: identify a peer group of banks (e.g., all banks in a single state) and pull comparable metrics across the group for one reporting period.

**Use case:** Comparing your institution's ratios against peers, benchmarking, regulatory analysis.

## Setup

In [ ]:
import os
import pandas as pd
from ffiec_data_connect import (
    OAuth2Credentials,
    collect_data,
    collect_filers_on_reporting_period,
    collect_reporting_periods,
)

creds = OAuth2Credentials(
    username=os.environ['FFIEC_USERNAME'],
    bearer_token=os.environ['FFIEC_BEARER_TOKEN'],
)

## Step 1: Get the panel of reporters for a period

In [ ]:
periods = collect_reporting_periods(creds, series='call')
# Use second-to-last period — latest may be a future quarter not yet filed
latest = periods[-2]
print(f'Using reporting period: {latest}')

panel = collect_filers_on_reporting_period(
    creds,
    reporting_period=latest,
    output_type='pandas',
)

print(f'Panel has {len(panel):,} institutions')
panel.head()

## Step 2: Filter to your peer group

In [ ]:
# Example: all banks in Rhode Island
TARGET_STATE = 'RI'

peers = panel[panel['state'] == TARGET_STATE].copy()
print(f'Found {len(peers)} banks in {TARGET_STATE}')
peers[['rssd', 'name', 'city', 'state']].head(10)

## Step 3: Pull Call Report data for each peer

This makes N API calls — one per peer. With the built-in rate limiter (2500 req/hr), you can comfortably handle a few hundred peers. For larger groups, consider running overnight or splitting the work.

In [ ]:
import time

peer_data = []

for idx, row in peers.iterrows():
    rssd = row['rssd']
    name = row['name']
    print(f'[{idx+1}/{len(peers)}] {rssd} — {name[:50]}', end=' ')
    try:
        df = collect_data(
            creds,
            reporting_period=latest,
            rssd_id=rssd,
            series='call',
            output_type='pandas',
        )
        df['rssd'] = rssd
        df['name'] = name
        peer_data.append(df)
        print(f'OK ({len(df)} items)')
    except Exception as e:
        print(f'FAIL: {type(e).__name__}')

all_peer_data = pd.concat(peer_data, ignore_index=True)
print(f'\nTotal rows: {len(all_peer_data):,}')

## Step 4: Extract comparable metrics

Pivot the long-format data into a wide comparison table. Each row is a bank, each column is a metric.

In [ ]:
# Common Call Report MDRM codes
METRICS = {
    'RCFD2170': 'total_assets',
    'RCFD2200': 'total_deposits',
    'RCFD2122': 'net_loans_leases',
    'RCFD3210': 'total_equity',
}

subset = all_peer_data[all_peer_data['mdrm'].isin(METRICS.keys())].copy()
subset['metric'] = subset['mdrm'].map(METRICS)

comparison = (
    subset.pivot_table(
        index=['rssd', 'name'],
        columns='metric',
        values='int_data',
        aggfunc='first',
    )
    .reset_index()
    .sort_values('total_assets', ascending=False)
)

# Convert from thousands to millions for readability
for col in METRICS.values():
    if col in comparison.columns:
        comparison[col] = comparison[col] / 1_000

comparison.head(10)

## Step 5: Compute derived ratios

In [ ]:
comparison['loans_to_deposits'] = (
    comparison['net_loans_leases'] / comparison['total_deposits']
)
comparison['equity_to_assets'] = (
    comparison['total_equity'] / comparison['total_assets']
)

comparison[['name', 'total_assets', 'loans_to_deposits', 'equity_to_assets']].head(10)

## Tips for larger peer groups

- **Hundreds of banks**: use `AsyncCompatibleClient` for parallel fetches (see the REST demo notebook)
- **Thousands of banks**: split the work into batches, checkpoint to disk, handle failures gracefully
- **State subsets**: the `panel` DataFrame has `state`, `city`, `fdic_cert_number`, and more — filter however you need